In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

class AbsMagnetization(qml.measurements.StateMeasurement):
    """A measurement class that estimates <|M|>."""

    def process_state(self, state, wire_order):
        """Calculates <|M|>.

        Args:
            state (Sequence[complex]): quantum state with a flat shape. It may also have an
                optional batch dimension.
            wire_order (Wires): wires determining the subspace that the state acts on; a matrix of
                dimension 2**n that acts on a subspace of n wires

        Returns: 
            abs_mag (float): <|M|>
        """
        if len(state.shape) == 2:  # (batch, dim)
            state = state[0]
        num_qubits = len(wire_order)
        abs_m = 0.0
        for idx in range(2**num_qubits):
            prob = np.abs(state[idx])**2
            mz = 0
            for i in range(num_qubits):
                mz += 1 if ((idx >> i) & 1) == 0 else -1
            abs_m += prob * np.abs(mz)
        # Return the average (per-site) magnetization
        return abs_m / num_qubits

def tfim_ground_state(num_qubits, h):
    """Calculates the ground state of the 1D TFIM Hamiltonian.
    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.
    Returns:
        (numpy.tensor): The ground state.
    """
    coeffs = []
    ops = []
    for i in range(num_qubits - 1):
        coeffs.append(-1.0)
        ops.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))
    for i in range(num_qubits):
        coeffs.append(-h)
        ops.append(qml.PauliX(i))
    H = qml.Hamiltonian(coeffs, ops)
    mat = qml.matrix(H, wire_order=range(num_qubits))
    eigvals, eigvecs = np.linalg.eigh(mat)
    ground_idx = np.argmin(eigvals)
    ground_state = eigvecs[:, ground_idx]
    return np.tensor(ground_state, requires_grad=False)

def get_magnetization_qnode(num_qubits):
    dev = qml.device("default.qubit", wires=num_qubits)
    @qml.qnode(dev)
    def magnetization(h):
        gs = tfim_ground_state(num_qubits, h)
        qml.StatePrep(gs, wires=range(num_qubits))
        return AbsMagnetization(wires=list(range(num_qubits)))
    return magnetization

def critical_point_estimate(mags, h_values):
    """Estimate the critical point by finding the point of most rapid change."""
    differences = [np.abs(mags[i] - mags[i + 1]) for i in range(len(mags) - 1)]
    ind = np.argmax(np.array(differences))
    h_c = np.mean([h_values[ind], h_values[ind + 1]])
    return h_c

def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    h_values = np.arange(0.2, 1.1, 0.005)
    mags = []
    magnetization = get_magnetization_qnode(num_qubits)
    for h in h_values:
        mags.append(magnetization(h))
    output = critical_point_estimate(np.array(mags), h_values)
    return str(np.round(output, 4))

def check(solution_output: str, expected_output: str) -> None:
    solution_output = float(solution_output)
    expected_output = float(expected_output)
    assert np.isclose(solution_output, expected_output, rtol=5e-3)

def test_abs_magnetization():
    # 2 qubits, |00>
    st = np.zeros(4)
    st[0] = 1.0
    proof = AbsMagnetization(wires=[0, 1])
    assert np.isclose(proof.process_state(st, [0, 1]), 1.0)
    # 2 qubits, |01>
    st = np.zeros(4)
    st[1] = 1.0
    assert np.isclose(proof.process_state(st, [0, 1]), 0.0)
    # GHZ state: (|00> + |11>)/sqrt(2)
    st = np.zeros(4, dtype=complex)
    st[0] = 1 / np.sqrt(2)
    st[3] = 1 / np.sqrt(2)
    # For both, mz = +2 or -2 => abs(mz)=2, prob=0.5 each, so <|M|> = 1
    assert np.isclose(proof.process_state(st, [0, 1]), 1.0)

def test_tfim_ground_state():
    # For h=0, the ground state should be |000..0> or |111..1>
    gs = tfim_ground_state(2, 0.0)
    # All amplitude on |00> or |11>
    assert np.isclose(np.abs(gs[0])**2 + np.abs(gs[3])**2, 1.0)
    gs5 = tfim_ground_state(3, 0.0)
    # All on |000> or |111>
    assert np.isclose(np.abs(gs5[0])**2 + np.abs(gs5[7])**2, 1.0)

def test_critical_point_estimate():
    mags = np.array([1.0, 0.8, 0.4, 0.01])
    hvals = np.array([0.2, 0.4, 0.6, 1.0])
    hc = critical_point_estimate(mags, hvals)
    assert np.isclose(hc, 0.5)

def test_run():
    out2 = run('2')
    assert np.isclose(float(out2), 0.3535, rtol=5e-3)
    out5 = run('5')
    assert np.isclose(float(out5), 0.6735, rtol=5e-3)

# Run all tests
if __name__ == "__main__":
    test_abs_magnetization()
    test_tfim_ground_state()
    test_critical_point_estimate()
    test_run()
    print("All tests passed.")

# The test cases from the prompt
test_cases = [
    ('5', '0.6735'),
    ('2', '0.3535')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
        continue

    try:
        check(output, expected_output)
    except AssertionError:
        print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
    else:
        print("Correct!")


All tests passed.
Running test case 0 with input '5'...


Correct!
Running test case 1 with input '2'...


Correct!
